In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'yt-dlp>=2024.11.18',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'pyloudnorm>=0.1.1',
    'pyarrow>=16.0.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)
# Rust extensions — install if .whl is available, skip gracefully otherwise
import glob as _glob
for _whl in _glob.glob('*.whl'):
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _whl], check=True)
        print(f'[install] installed {_whl}')
    except Exception as _e:
        print(f'[install] failed to install {_whl}: {_e}')

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2')

from shared.secrets import load_secrets
from shared.cf_client import CFClient
from shared.workflow_kernel import WorkflowKernel
from shared.repo_router import RepoRouter

import yaml
from pathlib import Path

CONFIG_DIR = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')
WORK_DIR   = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID       = os.environ.get('RUN_ID_OVERRIDE', 'run_20260507_001')
SESSION_ID   = 'cpu_collect_01'
SESSION_TYPE = 'cpu_collect'
SHARD_KEY    = 'cpu'

SECRETS = load_secrets(require_gemini=False)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO   = repos_cfg['repos']['stage0_codec']['repo_id']
OVERFLOW_REPO = repos_cfg['repos']['overflow']['repo_id']

kernel = WorkflowKernel(
    run_id           = RUN_ID,
    session_id       = SESSION_ID,
    session_type     = SESSION_TYPE,
    shard_key        = SHARD_KEY,
    cf_worker_url    = SECRETS['CF_WORKER_URL'],
    cf_worker_secret = SECRETS['CF_WORKER_SECRET'],
    gpu_type         = None,
    vram_limit_gb    = 0.0,
    session_max_hours = 8.5,
)
kernel.start()
print(f'[session] {SESSION_ID} started â€” run={RUN_ID}')

In [ ]:
from huggingface_hub import HfApi
import shutil

WORK_DIR       = Path('/kaggle/working')
DOWNLOAD_DIR   = WORK_DIR / 'raw_downloads'
STANDARD_DIR   = WORK_DIR / 'standardized'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1b.json'
FAILED_LOG     = WORK_DIR / 'failed_downloads.txt'
CONFIG_DIR     = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')
COOKIES_SRC    = CONFIG_DIR / 'cookies.txt'
COOKIES_PATH   = WORK_DIR / 'cookies.txt'

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
STANDARD_DIR.mkdir(parents=True, exist_ok=True)


if COOKIES_SRC.exists() and not COOKIES_PATH.exists():
    shutil.copy2(str(COOKIES_SRC), str(COOKIES_PATH))
    print(f'[config] cookies copied to {COOKIES_PATH}')

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'GEMINI_API_KEY':  c.get_secret('GEMINI_API_KEY_01') or c.get_secret('GEMINI_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass

    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')

    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS   = load_secrets()
HF_TOKEN  = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0 repo: {STAGE0_REPO}')
print(f'[config] cookies: {"found" if COOKIES_PATH.exists() else "NOT FOUND — 403s likely"}')

In [ ]:
import threading
import json
import requests
from datetime import datetime, timezone
import time

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — done={len(state["done"])} failed={len(state["failed"])} standardized={len(state["standardized"])}')
            return state
        except Exception:
            pass

    try:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1b.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback — done={len(state["done"])}')
            return state
    except Exception:
        pass

    print('[checkpoint] fresh start')
    return {
        'done': [],
        'failed': [],
        'standardized': [],
        'stats': {
            'downloaded': 0,
            'standardized': 0,
            'failed_download': 0,
            'failed_standardize': 0,
            'too_short': 0,
            'too_small': 0,
        },
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))

    if not upload:
        return

    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p1b.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1b checkpoint',
            )
            return
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[checkpoint] upload failed attempt {attempt+1}: {e} — retry in {wait}s')
            time.sleep(wait)


state = load_checkpoint()
done_set         = set(state['done'])
standardized_set = set(state['standardized'])

In [ ]:
import requests, json, time

manifest_local = WORK_DIR / 'video_manifest.jsonl'

BATCH_SIZE = 350

if not manifest_local.exists():
    print('[manifest] downloading from HF...')
    for attempt in range(6):
        try:
            url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/video_manifest.jsonl'
            r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=120, stream=True)
            r.raise_for_status()
            with open(manifest_local, 'wb') as f:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk)
            print(f'[manifest] downloaded to {manifest_local}')
            break
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[manifest] download attempt {attempt+1} failed: {e} — retry in {wait}s')
            time.sleep(wait)

_all_videos = []
with open(manifest_local, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            _all_videos.append(json.loads(line))

_initial_pending = [v for v in _all_videos if v['video_id'] not in done_set]
print(f'[manifest] total={len(_all_videos)} already_done={len(done_set)} initial_pending={len(_initial_pending)}')


In [ ]:
import json as _json
import shutil


def exec_notebook(path):
    with open(path) as _f:
        _nb = _json.load(_f)
    for _cell in _nb.get('cells', []):
        if _cell.get('cell_type') == 'code':
            _src = ''.join(_cell.get('source', []))
            exec(_src, globals())


def reload_checkpoint_state():
    cp = load_checkpoint()
    return set(cp['done']), set(cp['standardized']), cp['stats']


def purge_media_dirs():
    for d in [DOWNLOAD_DIR, STANDARD_DIR]:
        if d.exists():
            shutil.rmtree(str(d))
            d.mkdir(parents=True, exist_ok=True)
            print(f'[cleanup] purged {d}')


def count_pending():
    all_vids = []
    with open(manifest_local, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                all_vids.append(json.loads(line))
    current_done = set(load_checkpoint()['done'])
    return [v for v in all_vids if v['video_id'] not in current_done]


PIPELINE_DIR = '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/pipeline_1_collect'

repo_router = RepoRouter(STAGE0_REPO, OVERFLOW_REPO)

cycle = 0

while True:
    if kernel.session_expiring:
        print(f'[session] expiring before cycle {cycle + 1} — stopping')
        break

    remaining = count_pending()
    if not remaining:
        print(f'[session] all videos processed after {cycle} cycle(s) — done')
        break

    cycle += 1
    print(f'\n[session] === cycle {cycle} | pending={len(remaining)} ===')

    kernel.check_session_time()
    started_at = kernel.log_stage_start('p1a')
    try:
        exec_notebook(f'{PIPELINE_DIR}/p1a_discover.ipynb')
        kernel.log_stage_end('p1a', started_at)
        print(f'[session] p1a completed (cycle {cycle})')
    except Exception as e:
        kernel.log_stage_end('p1a', started_at, error=str(e))
        print(f'[session] p1a failed (cycle {cycle}): {e}')
        raise

    if kernel.session_expiring:
        print(f'[session] expiring after p1a cycle {cycle} — stopping')
        break

    remaining = count_pending()
    if not remaining:
        print(f'[session] no pending videos after p1a cycle {cycle} — done')
        break

    kernel.check_session_time()
    started_at = kernel.log_stage_start('p1b')
    try:
        exec_notebook(f'{PIPELINE_DIR}/p1b_download.ipynb')
        kernel.log_stage_end('p1b', started_at)
        print(f'[session] p1b completed (cycle {cycle})')
    except Exception as e:
        kernel.log_stage_end('p1b', started_at, error=str(e))
        print(f'[session] p1b failed (cycle {cycle}): {e}')
        raise

    done_set, standardized_set, _ = reload_checkpoint_state()

    if kernel.session_expiring:
        print(f'[session] expiring after p1b cycle {cycle} — stopping before p1e')
        break

    kernel.check_session_time()
    started_at = kernel.log_stage_start('p1e')
    try:
        exec_notebook(f'{PIPELINE_DIR}/p1e_upload.ipynb')
        kernel.log_stage_end('p1e', started_at)
        print(f'[session] p1e completed (cycle {cycle})')
    except Exception as e:
        kernel.log_stage_end('p1e', started_at, error=str(e))
        print(f'[session] p1e failed (cycle {cycle}): {e}')
        raise

    purge_media_dirs()

    done_set, standardized_set, _ = reload_checkpoint_state()

    remaining_after = count_pending()
    print(f'[session] cycle {cycle} complete | remaining={len(remaining_after)}')

    if not remaining_after:
        print(f'[session] manifest fully processed in {cycle} cycle(s)')
        break

kernel.stop()
print('[session] cpu_collect session complete')


NameError: name 'RepoRouter' is not defined